In [ ]:
%pip install -q \
  faiss-cpu chromadb datasets tiktoken

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

In [ ]:
%pip install -q \
"cohere>=5.13"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 352.0/352.0 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 63.4 MB/s eta 0:00:00


In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
import faiss
import cohere
import getpass

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
PROJECT_PATH = "/content/drive/MyDrive/KeepCoding/Ai-Engineering/Practica"

DATA_PATH = os.path.join(PROJECT_PATH, "data")
os.makedirs(DATA_PATH, exist_ok=True)

In [ ]:
embeddings_path =os.path.join(DATA_PATH, "event_embeddings.pkl")
with open(embeddings_path, "rb") as f:
    all_embeddings = pickle.load(f)

In [ ]:
documents_path =os.path.join(DATA_PATH, "event_documents.pkl")
with open(documents_path, "rb") as f:
    documents = pickle.load(f)

In [ ]:
index = faiss.read_index(
    os.path.join(DATA_PATH, "faiss_index.bin")
)

In [ ]:
print(len(documents))
print(index.ntotal)

2500
2500


In [ ]:
if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key: ")

co = cohere.Client(os.environ["COHERE_API_KEY"])

Cohere API Key: ··········


In [ ]:
query = "Voy a hacer un tardeo universitario en Málaga para 1200 personas. ¿Qué precio y estrategia me recomiendas?"

En este paso genero el embedding de la consulta del usuario. A diferencia de los documentos, utilizo `input_type="search_query"` porque representa una búsqueda que se comparará con los documentos almacenados en FAISS.

Aunque solo haya una consulta, Cohere espera recibir una lista de textos (`texts=[query]`). Como únicamente devuelve un embedding, me quedo con el primero mediante `res.embeddings.float[0]`.


In [ ]:
res = co.embed(
    texts=[query],
    model="embed-multilingual-v3.0",
    input_type="search_query",
    embedding_types=["float"],
)
embs = res.embeddings.float[0]

Convertimos el embedding de la consulta al formato requerido por FAISS.

In [ ]:
query_vector = np.array([embs]).astype("float32")

k = 5

distances, indices = index.search(query_vector, k)

In [ ]:
print(indices)
print(distances)

[[ 468 1107   12  762  759]]
[[0.51945186 0.5272534  0.52761173 0.5567304  0.55791426]]


indices son las posiciones de los eventos más cercanos
y distancia cuanto más bajo más parecido

In [ ]:
for i in indices[0]:
    print("================================================================")
    print(documents[i])


  Evento: Tardeo universitario

  Ciudad: Málaga

  Tipo de recinto: Discoteca

  Género musical: Latino

  Público objetivo: Universitarios

  Día de la semana: Domingo

  Hora de inicio: 16:00

  Precio de entrada: 10 €

  Aforo: 1000 personas

  Ocupación: 96%

  Inicio de ventas: 14 días antes

  Estrategia de promoción: Preventa por tramos

  Canal principal de venta: Instagram

  Copas incluidas: 2

  Climatología: Frío

  Eventos competidores: 2

  Resultado del evento: Sold out o casi sold out

  Observaciones comerciales: Ventas lentas al inicio y aceleración final
  

  Evento: Tardeo universitario

  Ciudad: Málaga

  Tipo de recinto: Discoteca

  Género musical: Latino

  Público objetivo: Universitarios

  Día de la semana: Jueves

  Hora de inicio: 16:00

  Precio de entrada: 9 €

  Aforo: 3000 personas

  Ocupación: 99%

  Inicio de ventas: 7 días antes

  Estrategia de promoción: Promoción grupos

  Canal principal de venta: WhatsApp

  Copas incluidas: 2

  Climatolog

In [ ]:
query = "Voy a organizar un tardeo universitario en Madrid para unas 500 personas.Estoy pensando en poner un precio de entre 10 y 12 euros. ¿Qué estrategia de promoción suele funcionar mejor?"

In [ ]:
res = co.embed(
    texts=[query],
    model="embed-multilingual-v3.0",
    input_type="search_query",
    embedding_types=["float"],
)
embs = res.embeddings.float[0]

In [ ]:
query_vector = np.array([embs]).astype("float32")

k = 3

distances, indices = index.search(query_vector, k)

In [ ]:
for i in indices[0]:
    print("================================================================")
    print(documents[i])


  Evento: Tardeo universitario

  Ciudad: Madrid

  Tipo de recinto: Discoteca

  Género musical: Comercial

  Público objetivo: Universitarios

  Día de la semana: Domingo

  Hora de inicio: 18:00

  Precio de entrada: 12 €

  Aforo: 800 personas

  Ocupación: 88%

  Inicio de ventas: 10 días antes

  Estrategia de promoción: Comunidad WhatsApp

  Canal principal de venta: Web ticketing

  Copas incluidas: 2

  Climatología: Frío

  Eventos competidores: 0

  Resultado del evento: Buen resultado

  Observaciones comerciales: WhatsApp fue el canal más efectivo
  

  Evento: Tardeo universitario

  Ciudad: Madrid

  Tipo de recinto: Recinto exterior

  Género musical: Latino

  Público objetivo: Universitarios

  Día de la semana: Viernes

  Hora de inicio: 18:00

  Precio de entrada: 12 €

  Aforo: 600 personas

  Ocupación: 61%

  Inicio de ventas: 45 días antes

  Estrategia de promoción: Promoción grupos

  Canal principal de venta: WhatsApp

  Copas incluidas: 1

  Climatología: L

Tras realizar varias consultas de prueba, FAISS recupera los eventos más similares a la búsqueda realizada. Los primeros resultados coinciden con las características esperadas (tardeo universitario, Málaga, público universitario, precios y aforos similares), por lo que se puede considerar que el Retrieval funciona correctamente y el índice vectorial ha sido construido con éxito.
